# core

> Add/remove a GPU to/from your SolveIt instance in one function.

```mermaid
flowchart TD
    A[Provision Modal sandbox]
    B[Setup SSH connection]
    C[Link remote kernel]

    A --> B --> C
```

In [ ]:
#| default_exp core

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
enable_mermaid()

<script type="module">
if (window.mermaid) mermaid.run()
else {
    import('https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs').then(m => {
        window.mermaid = m.default;
        window.mermaid.run();
        htmx.onLoad(elt => {
            if (elt.matches('div.mermaid, pre.mermaid') || htmx.findAll(elt, 'div.mermaid, pre.mermaid')) window.mermaid.run();
        });
    });
}</script>

In [ ]:
#| export
from solveit_modal.modal import *
from solveit_modal.ipyfernel import *

In [ ]:
#| export
from ipyfernel.core import *

In [ ]:
#| export
_all_ = ['connect_existing_kernel', 'gip', 'ipf_exec', 'ipf_shutdown', 'ipf_startup',
         'local', 'register_remote_kernel', 'remote', 'set_ssh_config',
         'set_sticky', 'start_remote', 'stop_remote', 'unset_sticky']

In [ ]:
#| export
def provision_sandbox(
    gpu:str,        # GPU type string, e.g. 'T4', 'A100'
    timeout:int     # Max sandbox lifetime in seconds
) -> tuple:  # (sandbox, host, port)
    "Create Modal app, image, sandbox, and expose TCP tunnel."
    app = mk_app()
    img = mk_img()
    sb = mk_sandbox(app, img, gpu=gpu, timeout=timeout)
    host, port = get_tunnel(sb)
    return sb, host, port

In [ ]:
sb, host, port = provision_sandbox(gpu=None, timeout=1800); sb, host, port

∞ creating sandbox; this may take 5-10 minutes if you are creating this sandbox for the first time... - INFO | 2026-06-11 08:18:56,228


✔ sandbox ready - INFO | 2026-06-11 08:18:56,818


✔ gotten tunnel: r435.modal.host:36261 - INFO | 2026-06-11 08:18:58,181


(Sandbox(), 'r435.modal.host', 36261)

In [ ]:
#| export
import modal

In [ ]:
#| export
def setup_ssh(
    sb  :modal.sandbox.Sandbox,    # Modal Sandbox object
    host:str,                      # Tunnel hostname
    port:int                       # Tunnel port
) -> object:  # Paramiko SSH connection
    "Inject public key, start SSH service, and verify connectivity."
    inject_pubkey(sb, get_pubkey())
    start_ssh(sb)
    ssh = mk_ssh(host, port)
    verify_ssh(ssh)
    return ssh

In [ ]:
ssh = setup_ssh(sb, host, port); ssh

✔ public key injected - INFO | 2026-06-11 08:19:08,420


✔ started ssh service - INFO | 2026-06-11 08:19:08,902


System: Linux
Hostname: modal
User: root
Kernel: 4.4.0
Architecture: x86_64
OS Type: GNU/Linux
GPU: no GPU


<function solveit_modal.modal.mk_ssh.<locals>.<lambda>(*cmd)>

In [ ]:
#| export
import logging

In [ ]:
#| export
logging.basicConfig(level=logging.INFO, format='%(message)s - %(levelname)s | %(asctime)s')
log = logging.getLogger(__name__)

In [ ]:
#| export
def link_remote_kernel(
    ssh   :object,     # Paramiko SSH connection
    host  :str,        # Tunnel hostname
    port  :int,        # Tunnel port
    sticky:bool=False  # Set `True` to execute code remotely by default
):
    "Launch remote IPython kernel on sandbox and connect ipyfernel to it."
    start_remote_kernel(ssh)
    p = get_remote_python(ssh)
    register_remote_kernel(remote_python=p)
    connect_existing_kernel(f'{host}:{port}')
    log.info('✔ connected to remote kernel')
    if sticky: 
        set_sticky()
        log.warning('! code will run remotely by default; run `%local unset_sticky` to run code locally')

In [ ]:
link_remote_kernel(ssh, host, port)

∞ starting kernel - INFO | 2026-06-11 08:20:10,726


✔ remote kernel ready: /root/.local/share/jupyter/runtime/kernel-ipyf.json - INFO | 2026-06-11 08:20:13,766


ipyf_remote_kernel is already a registered kernel
/app/data/.ssh/config file updated.


Successfully created connection file and forwarded ports!


✔ connected to remote kernel - INFO | 2026-06-11 08:20:15,466


Success: connected to remote kernel via r435.modal.host:36261


In [ ]:
stop_remote()

Code cells will now run locally.
Shutting down remote kernel


In [ ]:
sb.terminate()

In [ ]:
#| export
def gpu_on(
      gpu='T4',    # GPU type string
      timeout=1800 # Max sandbox lifetime in seconds
) -> tuple:        # (sandbox, ssh)
    "Provision a GPU sandbox, wire up SSH, and hijack cells onto a remote kernel."
    sb, host, port = provision_sandbox(gpu, timeout)
    ssh = setup_ssh(sb, host, port)
    link_remote_kernel(ssh, host, port)
    return sb, ssh

In [ ]:
sb, _ = gpu_on()

∞ creating sandbox; this may take 5-10 minutes if you are creating this sandbox for the first time... - INFO | 2026-06-11 08:22:06,715


✔ sandbox ready - INFO | 2026-06-11 08:22:07,217


✔ gotten tunnel: r431.modal.host:40943 - INFO | 2026-06-11 08:22:19,866


✔ public key injected - INFO | 2026-06-11 08:22:20,128


✔ started ssh service - INFO | 2026-06-11 08:22:20,694


∞ starting kernel - INFO | 2026-06-11 08:22:21,828


System: Linux
Hostname: modal
User: root
Kernel: 4.4.0
Architecture: x86_64
OS Type: GNU/Linux
GPU: Tesla T4


✔ remote kernel ready: /root/.local/share/jupyter/runtime/kernel-ipyf.json - INFO | 2026-06-11 08:22:24,613


✔ connected to remote kernel - INFO | 2026-06-11 08:22:25,127


ipyf_remote_kernel is already a registered kernel
connect_existing_kernel: already connected


In [ ]:
%remote !nvidia-smi

Thu Jun 11 08:21:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:18:00.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#| export
def gpu_off(
      sb  # sandbox object from gpu_on
):
    "Restore local kernel, disconnect remote, and teardown sandbox. Must be called from a %%local cell."
    unset_sticky()
    stop_remote()
    log.info('✔ unlinked from remote kernel')
    sb.terminate()
    log.info('✔ terminated sandbox')

In [ ]:
gpu_off(sb)

✔ unlinked from remote kernel - INFO | 2026-06-11 08:22:31,626


✔ terminated sandbox - INFO | 2026-06-11 08:22:31,687


Code cells will now run locally.
Code cells will now run locally.
Shutting down remote kernel


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()